# Preprocessing + model training + validation + Kaggle submission generation

In [33]:
from pathlib import Path
 
import pandas as pd
import os
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

In [34]:
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

TRAIN_PATH = "./dataset/train.csv"
TEST_PATH = "./dataset/test.csv"

OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_NUMERIC_FEATURES = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

BASE_CATEGORICAL_FEATURES = [
    "Sex",
    "Embarked"
]

ENGINEERED_NUMERIC_FEATURES = [
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
    "AgePclass"
]

ENGINEERED_CATEGORICAL_FEATURES = [
    "Title",
    "Deck",
    "FamilySizeGroup",
    "TicketPrefix"
]
 
MODELS = {
    "logreg": LogisticRegression(max_iter=1000),
    "rf": RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=3,
        random_state=42, n_jobs=-1,
    ),
    "gb": GradientBoostingClassifier(random_state=42),
}

# Feature Extractor

In [35]:
def add_features(df):
    df = df.copy()

    # Family features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    df["FamilySizeGroup"] = pd.cut(
        df["FamilySize"],
        bins=[0, 1, 4, 7, 20],
        labels=["Alone", "Small", "Medium", "Large"]
    )

    # Name features
    df["Title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
    )

    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    rare_titles = [
        "Lady", "Countess", "Capt", "Col", "Don",
        "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"
    ]

    df["Title"] = df["Title"].replace(
        {title: "Rare" for title in rare_titles}
    )

    # Cabin / deck
    df["Deck"] = df["Cabin"].str[0].fillna("U")

    # Ticket features
    df["TicketPrefix"] = (
        df["Ticket"]
        .str.replace(r"\d", "", regex=True)
        .str.replace(r"[\s./]+", "", regex=True)
        .replace("", "NONE")
    )

    # Fare features
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

    # Interaction
    df["AgePclass"] = df["Age"] * df["Pclass"]

    return df


def get_feature_lists(use_engineered: bool = True):
    """Return (numeric_features, categorical_features) lists."""
    numeric = list(BASE_NUMERIC_FEATURES)
    categorical = list(BASE_CATEGORICAL_FEATURES)
    if use_engineered:
        numeric += ENGINEERED_NUMERIC_FEATURES
        categorical += ENGINEERED_CATEGORICAL_FEATURES
    return numeric, categorical

In [27]:
def build_pipeline(model_key: str, numeric_features, categorical_features) -> Pipeline:
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MODELS[model_key]),
    ])

In [28]:
def load_data(use_engineered: bool):
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    if use_engineered:
        train = add_features(train)
        test = add_features(test)
    return train, test

In [29]:
def run(model_key="logreg", engineered=True, cv=5):
    print(f"Model: {model_key}")
    print(f"Engineered features: {engineered}")

    # Features
    numeric_features, categorical_features = get_feature_lists(use_engineered=engineered)
    all_features = numeric_features + categorical_features
    print(f"Features used: {all_features}\n")

    # Data
    train, test = load_data(engineered)
    X = train[all_features]
    y = train["Survived"]

    # Pipeline
    pipeline = build_pipeline(model_key, numeric_features, categorical_features)

    # Hold-out validation
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    pipeline.fit(X_tr, y_tr)
    val_pred = pipeline.predict(X_va)
    val_acc = accuracy_score(y_va, val_pred)

    print("=" * 60)
    print(f"Hold-out validation accuracy: {val_acc:.4f}")
    print("=" * 60)
    print(classification_report(y_va, val_pred))

    # Cross-validation
    cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
    print(f"{cv}-fold CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"Fold scores: {cv_scores.round(4).tolist()}\n")

    # Refit on all train → predict test → submission
    pipeline.fit(X, y)
    test_pred = pipeline.predict(test[all_features])

    submission = pd.DataFrame({
        "PassengerId": test["PassengerId"],
        "Survived": test_pred,
    })
    sub_path = OUT_DIR / f"submission_{model_key}.csv"
    submission.to_csv(sub_path, index=False)

    print(f"Submission saved to: {sub_path}")
    print(submission.head())
    print("Rows:", submission.shape[0], "(should be 418)")

    return pipeline, val_acc, cv_scores.mean()

In [30]:
if __name__ == "__main__":
    run()

Model: logreg
Engineered features: True
Features used: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'AgePclass', 'Sex', 'Embarked', 'Title', 'Deck', 'FamilySizeGroup', 'TicketPrefix']

Hold-out validation accuracy: 0.8156
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       110
           1       0.76      0.75      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.82      0.82      0.82       179

5-fold CV accuracy: 0.8272 (+/- 0.0232)
Fold scores: [0.8324, 0.8315, 0.7978, 0.809, 0.8652]

Submission saved to: outputs\submission_logreg.csv
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
Rows: 418 (should be 418)


# cross-validation + hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

numeric_features, categorical_features = get_feature_lists(use_engineered=True)

train, test = load_data(use_engineered=True)

all_features = numeric_features + categorical_features

X = train[all_features]
y = train["Survived"]

pipeline = build_pipeline(
    "rf",
    numeric_features,
    categorical_features
)

param_grid = {
    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [4, 6, 8, None],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("Best CV score:", grid.best_score_)
print("Best parameters:", grid.best_params_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best CV score: 0.8372481325717155
Best parameters: {'classifier__max_depth': 6, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 500}


In [ ]:
# Best model found by GridSearchCV
best_model = grid.best_estimator_

# Train on all training data
best_model.fit(X, y)

test_predictions = best_model.predict(test[all_features])

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

submission.to_csv("outputs/submission_rf_tuned.csv", index=False)

print(submission.head())
print("Rows:", len(submission))

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
Rows: 418
